<a href="https://colab.research.google.com/github/kadirrkirac/material-informatics/blob/main/sofc%20proje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install mp-api matminer pymatgen scikit-learn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 MB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [1]:
import pandas as pd
import numpy as np
from mp_api.client import MPRester
from matminer.featurizers.conversions import StrToComposition
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition
from sklearn.ensemble import RandomForestRegressor
from google.colab import userdata

# =================================================================
# 1. VERİ TOPLAMA (Materials Project API)
# =================================================================
def fetch_perovskite_data():
    api_key = userdata.get('MP_API_KEY')
    with MPRester(api_key) as mpr:
        print("Materials Project'ten veriler çekiliyor...")
        # Perovskit tabanlı (*O3) yapıları ve oluşum enerjilerini sorgula
        docs = mpr.summary.search(
            formula="*O3",
            fields=["formula_pretty", "formation_energy_per_atom"]
        )
        df = pd.DataFrame([
            {"Formula": d.formula_pretty, "Energy": d.formation_energy_per_atom}
            for d in docs
        ])
        print(f"Toplam {len(df)} adet referans veri seti başarıyla indirildi.")
        return df

# =================================================================
# 2. ÖZELLİK MÜHENDİSLİĞİ & MODEL EĞİTİMİ
# =================================================================
def train_stability_model(df):
    # Formülü kimyasal kompozisyona çevir
    stc = StrToComposition()
    df = stc.featurize_dataframe(df, "Formula")

    # Magpie preset'i kullanarak fiziksel özellikleri (yarıçap, elektronegatiflik vb.) ekle
    ep = ElementProperty.from_preset(preset_name="magpie")
    df = ep.featurize_dataframe(df, col_id="composition")

    # Sayısal olmayan sütunları ve hedef değişkeni ayır
    X = df.select_dtypes(include=[np.number]).drop(['Energy'], axis=1)
    y = df['Energy']

    # Model: Random Forest Regressor
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X, y)
    print("Makine öğrenmesi modeli eğitildi.")
    return model, X.columns, stc, ep

# =================================================================
# 3. HİBRİT TAHMİN FONKSİYONU (ML + Geometri)
# =================================================================
def predict_stability(formula, model, feature_columns, stc, ep):
    try:
        # Girdi hazırlama
        t_df = pd.DataFrame({"Formula": [formula]})
        t_df = stc.featurize_dataframe(t_df, "Formula", ignore_errors=True)
        t_df = ep.featurize_dataframe(t_df, col_id="composition")

        # Eğitim setindeki sütun yapısına eşitleme
        features = pd.DataFrame(0, index=[0], columns=feature_columns)
        for col in feature_columns:
            if col in t_df.columns:
                features[col] = t_df[col].values[0]

        # ML Enerji Tahmini
        raw_energy = model.predict(features)[0]

        # Goldschmidt Tolerans Faktörü (t) Hesabı
        comp = Composition(formula)
        elements = [el for el in comp.elements if el.symbol != 'O']
        radii = [el.atomic_radius for el in elements if el.atomic_radius is not None]

        if len(radii) >= 2:
            radii = sorted(radii, reverse=True)
            r_A, r_B = radii[0], radii[1]
            t = (r_A + 1.40) / (np.sqrt(2) * (r_B + 1.40))
        else:
            t = 0.0 # Geçersiz geometri (Soygazlar vb.)

        # Karar Mekanizması (Eşik değerleri optimize edilmiştir)
        if t >= 0.82 and raw_energy < -1.55:
            status = "✅ KESİN KARARLI"
        elif (0.75 <= t <= 1.10) or raw_energy < -1.45:
            status = "⚠️ SINIRDA / METASTABİL"
        else:
            status = "❌ RİSKLİ / KARARSIZ"

        return {
            "Formula": formula,
            "Energy": round(raw_energy, 4),
            "t_factor": round(t, 3),
            "Result": status
        }
    except Exception as e:
        return {"Formula": formula, "Error": str(e)}

# =================================================================
# 4. ÇALIŞTIRMA & ANALİZ
# =================================================================
if __name__ == "__main__":
    # Veri ve Model hazırlığı
    raw_df = fetch_perovskite_data()
    model, features, stc, ep = train_stability_model(raw_df)

    # Test Listesi (Proje ve Referanslar)
    test_list = [
        "BaTi1O3",                      # Referans 1
        "LaAl1O3",                      # Referans 2
        "Sr0.7 Ca0.2 Sm0.1 Co O3",       # TÜBİTAK Proje Formülü
        "Sr0.9 Sm0.1 Co O3",            # Optimize Formül
        "He1 Ne1 O3"                    # Negatif Kontrol
    ]

    print("\n" + "="*60)
    print(f"{'FORMÜL':<25} | {'ENERJİ':<8} | {'t':<5} | {'SONUÇ'}")
    print("-" * 60)

    for f in test_list:
        res = predict_stability(f, model, features, stc, ep)
        if "Error" in res:
            print(f"{res['Formula']:<25} | HATA: {res['Error']}")
        else:
            print(f"{res['Formula']:<25} | {res['Energy']:<8} | {res['t_factor']:<5} | {res['Result']}")
    print("="*60)

Materials Project'ten veriler çekiliyor...


/tmp/ipykernel_3934/144758943.py:18: DeprecationWarning: Accessing summary data through MPRester.summary is deprecated. Please use MPRester.materials.summary instead.
  docs = mpr.summary.search(


Retrieving SummaryDoc documents:   0%|          | 0/182 [00:00<?, ?it/s]

Toplam 182 adet referans veri seti başarıyla indirildi.


StrToComposition:   0%|          | 0/182 [00:00<?, ?it/s]

...

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

Sr0.9 Sm0.1 Co O3         | -1.4935  | 0.74  | ⚠️ SINIRDA / METASTABİL


StrToComposition:   0%|          | 0/1 [00:00<?, ?it/s]

ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

He1 Ne1 O3                | -0.1867  | 0.0   | ❌ RİSKLİ / KARARSIZ
